# Assignment 5 — Frequency-Domain Enhancement

Implements Gaussian low-pass filtering and manual gradient computation.

This notebook is intentionally modular. **Style 1** uses OpenCV operations; **Style 2** uses OpenCV only for reading and Matplotlib for display, while the image-processing logic is written with arrays, loops, queues, and formulas.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

LAB_DIR = Path.cwd()

def show_images(images, titles, cmap="gray", figsize=(14, 4)):
    """Display a sequence of images consistently in a notebook."""
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    axes = np.atleast_1d(axes)
    for axis, image, title in zip(axes, images, titles):
        axis.imshow(image, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()


## Style 1 — OpenCV built-in operations

In [ ]:
def gaussian_low_pass_cv(image, cutoff=30):
    spectrum = cv2.dft(image.astype(np.float32), flags=cv2.DFT_COMPLEX_OUTPUT)
    rows, cols = image.shape
    mask = np.zeros((rows, cols), np.float32)
    cv2.circle(mask, (cols // 2, rows // 2), cutoff, 1, -1)
    filtered = spectrum * mask[..., None]
    restored = cv2.idft(filtered, flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)
    return np.clip(restored, 0, 255).astype(np.uint8), mask

def butterworth_high_pass_cv(image, cutoff=30, order=2):
    rows, cols = image.shape
    y, x = np.ogrid[:rows, :cols]
    distance = np.hypot(x - cols / 2, y - rows / 2)
    mask = 1 / (1 + (cutoff / np.maximum(distance, 1e-6)) ** (2 * order))
    spectrum = cv2.dft(image.astype(np.float32), flags=cv2.DFT_COMPLEX_OUTPUT)
    result = cv2.idft(spectrum * mask[..., None], flags=cv2.DFT_SCALE | cv2.DFT_REAL_OUTPUT)
    return np.clip(result, 0, 255).astype(np.uint8), mask


## Style 2 — DSA/manual implementation

In [ ]:
def gaussian_low_pass_manual(image, cutoff=30):
    rows, cols = image.shape; spectrum = np.fft.fftshift(np.fft.fft2(image))
    mask = np.zeros((rows, cols), float)
    for r in range(rows):
        for c in range(cols):
            distance = ((r - rows / 2) ** 2 + (c - cols / 2) ** 2) ** 0.5
            mask[r, c] = np.exp(-(distance ** 2) / (2 * cutoff ** 2))
    result = np.real(np.fft.ifft2(np.fft.ifftshift(spectrum * mask)))
    return np.clip(result, 0, 255).astype(np.uint8), mask

def sobel_manual(image):
    gx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
    gy = gx.T; padded = np.pad(image.astype(float), 1, mode="edge"); result = np.zeros_like(image, float)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            w = padded[r:r + 3, c:c + 3]; result[r, c] = np.hypot((w * gx).sum(), (w * gy).sum())
    return np.clip(result, 0, 255).astype(np.uint8)


## Demonstration

In [ ]:
image = cv2.imread(str(LAB_DIR / "building.png"), cv2.IMREAD_GRAYSCALE)
if image is None: image = np.tile(np.arange(256, dtype=np.uint8), (256, 1))
image = cv2.resize(image, (256, 256))
filtered, mask = gaussian_low_pass_cv(image); manual, _ = gaussian_low_pass_manual(image)
print("Low-pass outputs:", filtered.shape, manual.shape)
show_images([image, filtered, manual], ["Input", "Gaussian LPF (Style 1)", "Gaussian LPF (Style 2)"])
